In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from math import radians, sin, cos, sqrt, atan2

In [3]:
df = pd.read_csv("nyc_bus_cleaned.csv")

df["RecordedAtTime"] = pd.to_datetime(df["RecordedAtTime"], errors="coerce")
df["ExpectedArrivalTime"] = pd.to_datetime(df["ExpectedArrivalTime"], errors="coerce")
df["Scheduled_dt"] = pd.to_datetime(df["Scheduled_dt"], errors="coerce")

def normalize_stop_name(s):
    if pd.isna(s):
        return np.nan
    s = str(s).upper().strip()
    s = " ".join(s.split())
    return s

df["stop_name"] = df["NextStopPointName"].apply(normalize_stop_name)
df["route"] = df["PublishedLineName"]
df["direction"] = df["DirectionRef"]

In [4]:
valid_proximity = ["at stop", "approaching"]
df5c = df[df["ArrivalProximityText"].isin(valid_proximity)].copy()

df5c = df5c.dropna(subset=[
    "route", "direction", "stop_name",
    "VehicleLocation.Latitude", "VehicleLocation.Longitude"
])

In [5]:
route_edges_df = pd.read_csv("graph_edges.csv")

route_edges_df = route_edges_df.rename(columns={
    "from_stop": "from_stop",
    "to_stop": "to_stop",
    "route": "route",
    "direction": "direction",
    "distance_m": "distance_m",
    "from_obs": "from_obs",
    "to_obs": "to_obs"
})

route_edges_df.head()

,route,direction,from_stop,to_stop,from_lat,from_lon,to_lat,to_lon,distance_m,from_obs,to_obs
0,B35,0.0,39 ST/2 AV,39 ST/3 AV,40.655490,-74.010854,40.654138,-74.008598,242.559736,1971,1854
1,B35,0.0,39 ST/3 AV,39 ST/4 AV,40.654138,-74.008598,40.652815,-74.006473,231.866520,1854,1527
2,B35,0.0,39 ST/4 AV,39 ST/5 AV,40.652815,-74.006473,40.651442,-74.004200,245.107279,1527,2645
3,B35,0.0,39 ST/5 AV,39 ST/6 AV,40.651442,-74.004200,40.650020,-74.001847,253.784113,2645,553
4,B35,0.0,39 ST/6 AV,39 ST/7 AV,40.650020,-74.001847,40.648699,-73.999661,235.769930,553,821


In [9]:
route_delay_stats = (
    df5c.groupby(["route", "direction"], as_index=False)
    .agg(
        mean_delay_min=("delay_minutes", "mean"),
        median_delay_min=("delay_minutes", "median"),
        std_delay_min=("delay_minutes", "std"),
        n_delay_obs=("delay_minutes", "count")
    )
)

route_delay_stats["std_delay_min"] = route_delay_stats["std_delay_min"].fillna(0)
route_delay_stats.head()

,route,direction,mean_delay_min,median_delay_min,std_delay_min,n_delay_obs
0,B1,0.0,4.847473,2.050000,27.180943,54583
1,B1,1.0,6.187625,2.800000,21.430210,57861
2,B11,0.0,4.895051,2.933333,17.461220,41179
3,B11,1.0,5.753806,2.633333,15.326232,47862
4,B12,0.0,7.572333,3.733333,28.228320,69498


In [10]:
BUS_SPEED_KMH = 14.0
TRANSFER_PENALTY_MIN = 5.0

route_edges_df["sched_time_min"] = route_edges_df["distance_m"] / (BUS_SPEED_KMH * 1000 / 60)

route_edges_df = route_edges_df.merge(
    route_delay_stats,
    on=["route", "direction"],
    how="left"
)

for col in ["mean_delay_min", "median_delay_min", "std_delay_min", "n_delay_obs"]:
    if col in route_edges_df.columns:
        route_edges_df[col] = route_edges_df[col].fillna(0)

route_edges_df["risk_score"] = route_edges_df["std_delay_min"]
route_edges_df["service_strength"] = np.sqrt(
    route_edges_df["from_obs"].fillna(0) * route_edges_df["to_obs"].fillna(0)
)

route_edges_df["generalized_cost"] = (
    route_edges_df["sched_time_min"]
    + 0.5 * route_edges_df["mean_delay_min"].clip(lower=0)
    + 0.25 * route_edges_df["risk_score"]
)

In [11]:
route_edges_df.to_csv("graph_edges_enriched.csv", index=False)
route_edges_df.head()

,route,direction,from_stop,to_stop,from_lat,from_lon,to_lat,to_lon,distance_m,from_obs,to_obs,sched_time_min,mean_delay_min,median_delay_min,std_delay_min,n_delay_obs,risk_score,service_strength,generalized_cost
0,B35,0.0,39 ST/2 AV,39 ST/3 AV,40.655490,-74.010854,40.654138,-74.008598,242.559736,1971,1854,1.039542,9.407061,6.316667,22.819923,106575,22.819923,1911.605085,11.448053
1,B35,0.0,39 ST/3 AV,39 ST/4 AV,40.654138,-74.008598,40.652815,-74.006473,231.866520,1854,1527,0.993714,9.407061,6.316667,22.819923,106575,22.819923,1682.574813,11.402225
2,B35,0.0,39 ST/4 AV,39 ST/5 AV,40.652815,-74.006473,40.651442,-74.004200,245.107279,1527,2645,1.050460,9.407061,6.316667,22.819923,106575,22.819923,2009.705202,11.458971
3,B35,0.0,39 ST/5 AV,39 ST/6 AV,40.651442,-74.004200,40.650020,-74.001847,253.784113,2645,553,1.087646,9.407061,6.316667,22.819923,106575,22.819923,1209.415148,11.496158
4,B35,0.0,39 ST/6 AV,39 ST/7 AV,40.650020,-74.001847,40.648699,-73.999661,235.769930,553,821,1.010443,9.407061,6.316667,22.819923,106575,22.819923,673.804868,11.418954


In [13]:
routestops = pd.read_csv("route_stops_full.csv")

routestops = routestops.rename(columns={
    "PublishedLineName": "route",
    "DirectionRef": "direction",
    "Stop Name": "stop_name"
})

state_nodes = routestops[["route", "direction", "stop_name", "stoplat", "stoplon", "obs"]].copy()
state_nodes["state_id"] = (
    state_nodes["route"].astype(str) + "__" +
    state_nodes["direction"].astype(str) + "__" +
    state_nodes["stop_name"].astype(str)
)

state_nodes.head()

,route,direction,stop_name,stoplat,stoplon,obs,state_id
0,B35,0.0,14 AV/36 ST,40.641006,-73.982306,925,B35__0.0__14 AV/36 ST
1,B35,0.0,14 AV/39 ST,40.639481,-73.984300,1408,B35__0.0__14 AV/39 ST
2,B35,0.0,14 AV/CHURCH AV,40.641736,-73.981549,1037,B35__0.0__14 AV/CHURCH AV
3,B35,0.0,39 ST/10 AV,40.644695,-73.993030,784,B35__0.0__39 ST/10 AV
4,B35,0.0,39 ST/12 AV,40.642054,-73.988660,997,B35__0.0__39 ST/12 AV


In [14]:
state_edges = route_edges_df.copy()

state_edges["from_state"] = (
    state_edges["route"].astype(str) + "__" +
    state_edges["direction"].astype(str) + "__" +
    state_edges["from_stop"].astype(str)
)

state_edges["to_state"] = (
    state_edges["route"].astype(str) + "__" +
    state_edges["direction"].astype(str) + "__" +
    state_edges["to_stop"].astype(str)
)

state_edges["edge_type"] = "ride"
state_edges.head()

,route,direction,from_stop,to_stop,from_lat,from_lon,to_lat,to_lon,distance_m,from_obs,...,mean_delay_min,median_delay_min,std_delay_min,n_delay_obs,risk_score,service_strength,generalized_cost,from_state,to_state,edge_type
0,B35,0.0,39 ST/2 AV,39 ST/3 AV,40.655490,-74.010854,40.654138,-74.008598,242.559736,1971,...,9.407061,6.316667,22.819923,106575,22.819923,1911.605085,11.448053,B35__0.0__39 ST/2 AV,B35__0.0__39 ST/3 AV,ride
1,B35,0.0,39 ST/3 AV,39 ST/4 AV,40.654138,-74.008598,40.652815,-74.006473,231.866520,1854,...,9.407061,6.316667,22.819923,106575,22.819923,1682.574813,11.402225,B35__0.0__39 ST/3 AV,B35__0.0__39 ST/4 AV,ride
2,B35,0.0,39 ST/4 AV,39 ST/5 AV,40.652815,-74.006473,40.651442,-74.004200,245.107279,1527,...,9.407061,6.316667,22.819923,106575,22.819923,2009.705202,11.458971,B35__0.0__39 ST/4 AV,B35__0.0__39 ST/5 AV,ride
3,B35,0.0,39 ST/5 AV,39 ST/6 AV,40.651442,-74.004200,40.650020,-74.001847,253.784113,2645,...,9.407061,6.316667,22.819923,106575,22.819923,1209.415148,11.496158,B35__0.0__39 ST/5 AV,B35__0.0__39 ST/6 AV,ride
4,B35,0.0,39 ST/6 AV,39 ST/7 AV,40.650020,-74.001847,40.648699,-73.999661,235.769930,553,...,9.407061,6.316667,22.819923,106575,22.819923,673.804868,11.418954,B35__0.0__39 ST/6 AV,B35__0.0__39 ST/7 AV,ride


In [ ]:
TRANSFER_PENALTY_MIN = 5.0

transfer_rows = []

for stop_name, group in state_nodes.groupby("stop_name"):
    records = group[["state_id", "route", "direction", "stop_name"]].drop_duplicates().to_dict("records")

    for i in range(len(records)):
        for j in range(len(records)):
            if i == j:
                continue

            a = records[i]
            b = records[j]

            if a["route"] == b["route"]:
                continue

            transfer_rows.append({
                "route": "TRANSFER",
                "direction": -1,
                "from_stop": stop_name,
                "to_stop": stop_name,
                "from_state": a["state_id"],
                "to_state": b["state_id"],
                "distance_m": 0.0,
                "from_obs": 0.0,
                "to_obs": 0.0,
                "sched_time_min": 0.0,
                "mean_delay_min": 0.0,
                "median_delay_min": 0.0,
                "std_delay_min": 0.0,
                "n_delay_obs": 0,
                "risk_score": 0.0,
                "service_strength": 0.0,
                "generalized_cost": TRANSFER_PENALTY_MIN,
                "edge_type": "transfer"
            })

transfer_edges_df = pd.DataFrame(transfer_rows)

,route,direction,from_stop,to_stop,from_state,to_state,distance_m,from_obs,to_obs,sched_time_min,mean_delay_min,median_delay_min,std_delay_min,n_delay_obs,risk_score,service_strength,generalized_cost,edge_type
0,TRANSFER,-1,108 ST/WALDRON ST,108 ST/WALDRON ST,Q58__0.0__108 ST/WALDRON ST,Q58__1.0__108 ST/WALDRON ST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,5.0,transfer
1,TRANSFER,-1,108 ST/WALDRON ST,108 ST/WALDRON ST,Q58__1.0__108 ST/WALDRON ST,Q58__0.0__108 ST/WALDRON ST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,5.0,transfer
2,TRANSFER,-1,39 ST/10 AV,39 ST/10 AV,B35__0.0__39 ST/10 AV,B35__1.0__39 ST/10 AV,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,5.0,transfer
3,TRANSFER,-1,39 ST/10 AV,39 ST/10 AV,B35__1.0__39 ST/10 AV,B35__0.0__39 ST/10 AV,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,5.0,transfer
4,TRANSFER,-1,39 ST/12 AV,39 ST/12 AV,B35__0.0__39 ST/12 AV,B35__1.0__39 ST/12 AV,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,5.0,transfer


In [19]:
all_state_edges = pd.concat(
    [state_edges, transfer_edges_df],
    ignore_index=True,
    sort=False
)

G2 = nx.DiGraph()

for _, row in state_nodes.iterrows():
    G2.add_node(
        row["state_id"],
        stop_name=row["stop_name"],
        route=row["route"],
        direction=row["direction"],
        stoplat=row["stoplat"],
        stoplon=row["stoplon"],
        obs=row["obs"]
    )

for _, row in all_state_edges.iterrows():
    G2.add_edge(
        row["from_state"],
        row["to_state"],
        edge_type=row["edge_type"],
        route=row["route"],
        direction=row["direction"],
        from_stop=row["from_stop"],
        to_stop=row["to_stop"],
        distance_m=row["distance_m"],
        sched_time_min=row["sched_time_min"],
        mean_delay_min=row["mean_delay_min"],
        std_delay_min=row["std_delay_min"],
        risk_score=row["risk_score"],
        generalized_cost=row["generalized_cost"]
    )

print("State nodes:", G2.number_of_nodes())
print("State edges:", G2.number_of_edges())

State nodes: 540
State edges: 796


In [20]:
state_nodes.to_csv("layer2_state_nodes.csv", index=False)
all_state_edges.to_csv("layer2_state_edges.csv", index=False)
nx.write_gexf(G2, "layer2_routing_graph.gexf")

In [21]:
def candidate_states_for_stop(state_nodes_df, stop_name):
    return state_nodes_df.loc[state_nodes_df["stop_name"] == stop_name, "state_id"].tolist()

def best_path_between_stops(G, state_nodes_df, origin_stop, destination_stop, weight="generalized_cost"):
    origins = candidate_states_for_stop(state_nodes_df, origin_stop)
    destinations = candidate_states_for_stop(state_nodes_df, destination_stop)

    best_result = None

    for o in origins:
        for d in destinations:
            if o == d:
                continue
            try:
                path = nx.shortest_path(G, o, d, weight=weight)
                cost = nx.shortest_path_length(G, o, d, weight=weight)

                if best_result is None or cost < best_result["cost"]:
                    best_result = {
                        "origin_state": o,
                        "destination_state": d,
                        "path": path,
                        "cost": cost
                    }
            except nx.NetworkXNoPath:
                continue

    return best_result

In [30]:
best_result = best_path_between_stops(
    G2,
    state_nodes,
    origin_stop="14 AV/36 ST",
    destination_stop="39 ST/12 AV"
)

best_result["path"]

['B35__0.0__14 AV/36 ST',
 'B35__0.0__14 AV/CHURCH AV',
 'B35__0.0__CHURCH AV/MC DONALD AV',
 'B35__1.0__CHURCH AV/MC DONALD AV',
 'B35__1.0__CHURCH AV/STORY ST',
 'B35__1.0__CHURCH AV/CHESTER AV',
 'B35__1.0__13 AV/37 ST',
 'B35__1.0__39 ST/13 AV',
 'B35__1.0__39 ST/12 AV']